# SMS Spam Detector | Colab training

**Goal:** train a simple model that labels an SMS as spam or ham (legitimate). Run each cell from top to bottom. This notebook uses TF-IDF to turn words into numbers and LinearSVC to classify them.

## 1. Install the small set of libraries
Colab already includes most of these. `joblib` saves the trained model for the website.

In [ ]:
!pip -q install scikit-learn joblib matplotlib

## 2. Download and read the public SMS Spam Collection
The dataset has one label and one message per row. `ham` means normal; `spam` means unwanted.

In [ ]:
import zipfile, urllib.request, pathlib
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "/content/sms.zip"
urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:
    data_file = next(n for n in z.namelist() if n.endswith("SMSSpamCollection"))
    z.extract(data_file, "/content")

import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("/content/SMSSpamCollection", sep="\t", header=None, names=["label", "message"])
print("Rows:", len(df))
display(df.head())
label_counts = df["label"].value_counts().reindex(["ham", "spam"])
print(label_counts)

# Graph 1: show how many normal and spam messages are in the dataset.
ax = label_counts.plot(kind="bar", color=["#159d8c", "#f07855"], figsize=(6, 4))
ax.set_title("SMS dataset: ham and spam messages")
ax.set_xlabel("Message label")
ax.set_ylabel("Number of messages")
ax.set_xticklabels(["Ham (normal)", "Spam"], rotation=0)
plt.tight_layout()
plt.show()


## 3. Split, train, and measure
We keep 20% aside as test data. The model never sees those messages during training, so the score estimates performance on unseen examples. `stratify` preserves the spam/ham ratio.

We add a small set of hand-written spam examples to the **training data only**. The original held-out test messages remain untouched, so the reported score is measured on the UCI test split. These examples are synthetic and help demonstrate how additional labeled examples can teach the model about newer wording.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
# Extra hand-written examples: add only to training data, never the held-out test set.
extra_spam_messages = [
    "You have won an iPhone. Claim your prize today!",
    "Congratulations, you have won a brand new iPhone!",
    "Winner alert: claim your free gift card now.",
    "You have been selected to receive a cash reward.",
    "Claim your prize before this limited offer expires.",
    "Exclusive deal: get a free phone today.",
    "You are today's lucky winner of a shopping voucher.",
    "Your number has won a holiday prize draw.",
    "Act now to claim your guaranteed reward.",
    "Special offer just for you: earn money from home fast.",
    "Free entry in our prize draw. Reply to claim your reward.",
    "Urgent: your reward is waiting. Claim it now.",
    "Congratulations! You won a luxury vacation package.",
    "Get a bonus gift when you respond today.",
    "You have won a supermarket voucher in our weekly draw.",
    "Limited time: claim your free cash bonus today.",
    "Your account qualifies for a free reward. Claim now.",
    "Winner! You have won a tablet in our promotion.",
    "Get cheap loans instantly. Apply now for fast approval.",
    "You have been chosen for an exclusive prize. Respond now.",
]
X_train = pd.concat([X_train, pd.Series(extra_spam_messages)], ignore_index=True)
y_train = pd.concat([y_train, pd.Series(["spam"] * len(extra_spam_messages))], ignore_index=True)
print(f"Added {len(extra_spam_messages)} synthetic spam examples to training data.")

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))),
    ("classifier", LinearSVC())
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Numeric evaluation on the held-out test messages.
accuracy = accuracy_score(y_test, predictions)
spam_precision = precision_score(y_test, predictions, pos_label="spam")
spam_recall = recall_score(y_test, predictions, pos_label="spam")
spam_f1 = f1_score(y_test, predictions, pos_label="spam")
print(f"Test accuracy:          {accuracy:.3%}")
print(f"Spam precision:         {spam_precision:.3%}")
print(f"Spam recall:            {spam_recall:.3%}")
print(f"Spam F1-score:          {spam_f1:.3%}")
print("\nFull classification report:")
print(classification_report(y_test, predictions, labels=["ham", "spam"], target_names=["ham", "spam"], digits=3))

# Graph 2: visualize the main evaluation scores.
score_names = ["Accuracy", "Spam precision", "Spam recall", "Spam F1"]
scores = [accuracy, spam_precision, spam_recall, spam_f1]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(score_names, scores, color=["#2864a5", "#159d8c", "#f0a23b", "#8b6cc1"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score (0 to 1)")
ax.set_title("Model performance on held-out test messages")
ax.bar_label(bars, labels=[f"{score:.1%}" for score in scores], padding=3)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

# Graph 3: confusion matrix; rows are actual labels, columns are predictions.
cm = confusion_matrix(y_test, predictions, labels=["ham", "spam"])
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Ham", "Spam"])
cm_display.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()


### How to explain these metrics in your viva

- **Accuracy:** the fraction of all test messages classified correctly.
- **Spam precision:** among messages predicted as spam, how many really are spam.
- **Spam recall:** among all actual spam messages, how many the model found.
- **Spam F1-score:** a single score balancing spam precision and spam recall.
- **Confusion matrix:** shows correct and incorrect ham/spam predictions; rows are actual labels and columns are predicted labels.

The charts use the held-out test split. The 20 hand-written examples are added only to training, so they do not become test examples. Use the exact scores printed by your Colab run in your report.


## 4. Try your own message

In [ ]:
def predict_message(message):
    result = model.predict([message])[0]
    return "SPAM" if result == "spam" else "HAM (normal)"

print(predict_message("Congratulations! You won a free prize. Call now!"))
print(predict_message("Are we still meeting at 5 today?"))

## 5. Save the model for the website
Download this file from Colab and place it beside `app.py` in the Antigravity project folder.

In [ ]:
import joblib
joblib.dump(model, "/content/spam_model.joblib")
from google.colab import files
files.download("/content/spam_model.joblib")